# 6. Effect of Slowly Time-Varying Currents

In [ ]:
import Pkg
Pkg.activate(joinpath(@__DIR__, ".."))
using LinkedMasses

Consider an ocean current rotating counterclockwise:

In [ ]:
function V_c_var(t::Real)
    T_c = 50.0 # Current time period
    arg = -2π * (t / T_c)
    return [-0.2, 0.0] + 0.05 * [cos(arg), sin(arg)]
end

The remaining parameters are identical to the baseline simulation:

In [ ]:
V_c_constant = [-0.2, 0.0]

m0 = 200.0
m = 40.0
c0 = 250.0
c = 50.0
L = 10.0

params = LinkedMassesParameters(L, m0, m, c0, c)

ε = 0.7

function path_circle(s::Real)    
    R = 10.0
    ϕ0 = -π / 2
    p_origin = [2.0, 2.0 + R]
    return p_origin + R * [cos(ϕ0 + s), sin(ϕ0 + s)]
end

U = 1.0
Δ = 5.0
los = LOSParameters(U, Δ)

k_v = 0.1
γ = 5.0

ctrl_var = AdaptiveLinearizingController(params, los, path_circle, k_v, γ, ε, V_c_var)
ctrl_const = AdaptiveLinearizingController(params, los, path_circle, k_v, γ, ε, V_c_constant)

In [ ]:
k_err = 1.5
params_estimate = LinkedMassesParameters(L, m0, m, k_err*c0, c/k_err)
ζ_hat0 = parameter_vector(params_estimate, ε, zeros(2))

using DifferentialEquations
T_stop = 300

# Initial state: both masses at rest, first mass at origin, second mass east of it
p0_init = [0.0, 0.0]
θ_init = -π / 2
v0_init = [0.0, 0.0]
ω_init = 0.0
s_init = 0.0
init_state = AdaptiveSimulationState(p0_init, θ_init, v0_init, ω_init, s_init, ζ_hat0)

sol_var = simulate(ctrl_var, init_state, T_stop; reltol=1e-6, saveat=2.0)
sol_const = simulate(ctrl_const, init_state, T_stop; reltol=1e-6)

In [ ]:
using Plots, LaTeXStrings, ColorSchemes
using CSV, DataFrames

scheme = ColorSchemes.Dark2_8.colors

folder = joinpath(@__DIR__, "csv")
mkpath(folder)
e_x, e_y, v_x, v_y = get_errors(sol_var, ctrl_var)
theta_dot = [u[6] for u in sol_var.u]

df_var = DataFrame(time=sol_var.t, e_x=e_x, e_y=e_y, v_x=v_x, v_y=v_y, theta_dot=theta_dot)
CSV.write(joinpath(folder, "adaptive_controller_variable_current.csv"), df_var)

e_x_const, e_y_const, v_x_const, v_y_const = get_errors(sol_const, ctrl_const)
theta_dot_const = [u[6] for u in sol_const.u]

fig1 = plot(sol_var.t, e_x, label=L"varying, $e_x$", xlabel="Time (s)", ylabel="Error (m)", legend=:topright, color=scheme[1], lw=2)
plot!(sol_var.t, e_y, label=L"varying, $e_y$", color=scheme[1], lw=2, ls=:dash)
plot!(sol_const.t, e_x_const, label=L"constant, $e_x$", color=scheme[2], lw=2)
plot!(sol_const.t, e_y_const, label=L"constant, $e_y$", color=scheme[2], lw=2, ls=:dash)

fig2 = plot(sol_var.t, v_x, label=L"varying, $v_x$", xlabel="Time (s)", ylabel="Velocity Error (m/s)", legend=:topright, color=scheme[1], lw=2)
plot!(sol_var.t, v_y, label=L"varying, $v_y$", color=scheme[1], lw=2, ls=:dash)
plot!(sol_const.t, v_x_const, label=L"constant, $v_x$", color=scheme[2], lw=2)
plot!(sol_const.t, v_y_const, label=L"constant, $v_y$", color=scheme[2], lw=2, ls=:dash)

fig3 = plot(sol_var.t, theta_dot, label="varying", xlabel="Time (s)", ylabel=L"\dot{\theta} (rad/s)", legend=:topright, color=scheme[1], lw=2)
plot!(sol_const.t, theta_dot_const, label="constant", color=scheme[2], lw=2)

s = [u[7] for u in sol_var.u]
p_path = hcat(path_circle.(s)...)'
fig4 = plot(p_path[:,2], p_path[:,1], title="Trajectory", label="Path", xlabel="East (m)", ylabel="North (m)", legend=:topright, color=:black, lw=2)

t_snapshots = [0.0, 45.0, 75.0, 100.0]
x0_snapshots = Float64[]
y0_snapshots = Float64[]
θ_snapshots = Float64[]
x_path_snapshots = Float64[]
y_path_snapshots = Float64[]
for t_snap in t_snapshots
    u_snap = sol_var(t_snap)
    push!(x0_snapshots, u_snap[1])
    push!(y0_snapshots, u_snap[2])
    push!(θ_snapshots, u_snap[3])
    push!(x_path_snapshots, path_circle(u_snap[7])[1])
    push!(y_path_snapshots, path_circle(u_snap[7])[2])
    plot_linked_masses!(fig4, u_snap, params; lw=1.5, marker=:o, ms=4, mc=scheme[1], lc=scheme[1], label="")
    scatter!(fig4, [path_circle(u_snap[7])[2]], [path_circle(u_snap[7])[1]], label="", ms=4, mc=:black, marker=:square)

    u_const = sol_const(t_snap)
    plot_linked_masses!(fig4, u_const, params; lw=1.5, marker=:o, ms=4, mc=scheme[2], lc=scheme[2], label="")
    scatter!(fig4, [path_circle(u_const[7])[2]], [path_circle(u_const[7])[1]], label="", ms=4, mc=:black, marker=:diamond)
end

df_snap = DataFrame(time = t_snapshots,
                    x0 = x0_snapshots,
                    y0 = y0_snapshots,
                    theta = θ_snapshots,
                    x_path = x_path_snapshots,
                    y_path = y_path_snapshots)
CSV.write(joinpath(folder, "adaptive_controller_snapshots_variable_current.csv"), df_snap)

plot(fig1, fig2, fig3, fig4, layout=(2,2), size=(900,600))
